## 1. Verifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer Ollama

In [ ]:
# zstd requis par l'installeur pour l'extraction ; pciutils (lspci) requis pour la détection GPU 
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2bis. configuration OLLAMA_MODELS
import os, glob

OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
READONLY_MODELS = False
print(f"{OLLAMA_MODELS_DIR} (téléchargement nécessaire ci-dessous).")

os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

## 3. Demarrer le serveur Ollama en arrière-plan

In [ ]:
import subprocess, time, requests

# Tuer tout ollama serve deja lancé sinon le nouveau process echoue au bind du port 11434 silencieusement.
!pkill -f "ollama serve" || true
time.sleep(2)  # laisser le temps au port de se liberer avant le nouveau bind

os.environ["OLLAMA_ORIGINS"] = "*" # désactive le controle CORS sur le header Origin. 

# Ecoute sur 0.0.0.0 : Ollama ne verifie plus le header Host des requetes (protection anti DNS-rebinding), ce qui rend inutile la reecriture du header cote tunnel (host_header n'existe plus dans l'agent ngrok v3).
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# la fenêtre de contexte par defaut d'Ollama (4096 tokens,prompt+completion CUMULES) coupe la génération en cours de synthese sur les questions multi-hop
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_LOAD_TIMEOUT"] = "30m"

OLLAMA_LOG_PATH = "/kaggle/working/ollama.log"  # un PIPE jamais lu se remplit et bloque le serveur ; le fichier garde aussi les erreurs des runners
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(OLLAMA_LOG_PATH, "w"), stderr=subprocess.STDOUT,
    env=os.environ,  # necessaire pour que OLLAMA_MODELS et OLLAMA_KEEP_ALIVE soient pris en compte
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama prêt.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama n'a pas demarré à temps -- lire OLLAMA_LOG_PATH.")

In [ ]:
# 3bis. Verifier qu'Ollama a bien detecté le GPU
!ollama --version
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 4. Télécharger les modèles

In [ ]:
MODEL = "qwen3:14b"

if not READONLY_MODELS:
    !ollama pull {MODEL}
else:
    print(f"{MODEL} supposé déja présent dans {OLLAMA_MODELS_DIR} -- vérification via 'ollama list' :")
    !ollama list

In [ ]:
# 4ter. Télécharger le modele d'embedding
EMBED_MODEL = "bge-m3"

if not READONLY_MODELS:
    !ollama pull {EMBED_MODEL}
else:
    print(f"{EMBED_MODEL} supposé deja présent dans {OLLAMA_MODELS_DIR}.")

## 5. Préchauffage + sanity check tool-calling

In [ ]:
# 5. Prechauffer le chat et l'embedding en VRAM avant le sanity check
# Ollama instancie un runner different selon la presence de 'tools'
import requests, time

OLLAMA_URL = "http://localhost:11434"

_t0 = time.time()
requests.post(
    f"{OLLAMA_URL}/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "hi"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Renvoie la météo pour une ville donnée.",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            },
        }],
        "stream": False,
    },
    timeout=600,
).raise_for_status()
print(f"Chat charge en VRAM ({time.time() - _t0:.1f}s).")

_t0 = time.time()
requests.post(
    f"{OLLAMA_URL}/v1/embeddings",
    json={"model": EMBED_MODEL, "input": "hi"},
    timeout=600,
).raise_for_status()
print(f"Embedding charge en VRAM ({time.time() - _t0:.1f}s).")

loaded = [entry["name"] for entry in requests.get(f"{OLLAMA_URL}/api/ps", timeout=30).json()["models"]]
print("Modeles en VRAM :", loaded)
assert any(name.startswith(MODEL) for name in loaded), f"{MODEL} a ete decharge."
assert any(name.startswith(EMBED_MODEL) for name in loaded), (
    f"{EMBED_MODEL} a decharge le chat : verifier OLLAMA_MAX_LOADED_MODELS (cellule 7)."
)

In [ ]:
# 5bis. Sanity check local -- verifier que le tool-calling natif fonctionne

import requests, json as _json

test_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Quel temps fait-il a Paris ?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Renvoie la meteo pour une ville donnée.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }],
    "stream": False,
}

resp = requests.post("http://localhost:11434/v1/chat/completions", json=test_payload, timeout=300)  # large par securité, le modèle est déja préchauffé
resp.raise_for_status()
data = resp.json()
tool_calls = data["choices"][0]["message"].get("tool_calls")
print("Tool calls recus :", _json.dumps(tool_calls, indent=2, ensure_ascii=False))
assert tool_calls, "tool-calling natif KO, ne pas continuer avec ce modèle."
print("tool-calling natif confirmé.")

## 6. Exposer le serveur via un tunnel public

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
conf.get_default().auth_token = user_secrets.get_secret("NGROK_AUTHTOKEN")

# Pas de host_header : l'agent ngrok v3 le refuse ("field host_header not found in type config.Endpoint").
# Ollama ecoute sur 0.0.0.0 (cellule 6) et n'exige donc plus un header Host local.
for _tunnel in ngrok.get_tunnels():  # relancer la cellule sans fermer l'ancien tunnel echoue sinon
    ngrok.disconnect(_tunnel.public_url)
public_url = ngrok.connect(11434, "http").public_url
print("URL publique ngrok (temporaire, valable tant que ce notebook tourne) :")
print(public_url)

import requests
_resp = requests.get(f"{public_url}/v1/models", timeout=30)
print("Test via le tunnel :", _resp.status_code)
if _resp.status_code != 200:
    raise RuntimeError(f"Le tunnel repond {_resp.status_code} : {_resp.text[:300]}")
print("Modeles servis :", [m["id"] for m in _resp.json()["data"]])
print("\nA reporter dans le .env du poste local :")
print(f"OLLAMA_HOST={public_url}")
print(f"KAGGLE_OLLAMA_HOST={public_url}   # avec LLM_TARGET=kaggle dans le .env")


In [ ]:
6*9